In [2]:
from langsmith import evaluate
from finrag.eval import target, hit_rate, evidence_recall, correctness

results = evaluate(
    target,
    data="financebench-dev",
    evaluators=[hit_rate,evidence_recall,correctness],
    experiment_prefix="baseline-pages-title",
    max_concurrency=1,
)

View the evaluation results for experiment: 'baseline-pages-title-4b15f13c' at:
https://smith.langchain.com/o/801c5b2a-1893-4c54-8fa4-d71735b292c9/datasets/7f47c824-9827-47e3-b3bb-2ae38d571f09/compare?selectedSessions=a83bbc5a-6278-49e7-9876-fb1b4e050bb5




0it [00:00, ?it/s]

Loading embeddinggemma2 on cuda 


Loading weights: 100%|██████████| 1376/1376 [00:01<00:00, 782.86it/s]
51it [06:59,  8.23s/it]


In [2]:
from langfuse import get_client
from finrag.eval_langhfuse import task, hit_rate, evidence_recall, correctness, average_scores

langfuse = get_client()
dataset = langfuse.get_dataset("financebench-dev")

result = dataset.run_experiment(
    name="baseline-pages-title",
    task=task,
    evaluators=[hit_rate, evidence_recall, correctness],
    run_evaluators=[average_scores],
    max_concurrency=1,
)

print(result.format())
langfuse.flush()

Individual Results: Hidden (51 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: baseline-pages-title
📋 Run name: baseline-pages-title - 2026-11-06T10:13:31.865742Z
51 items
Evaluations:
  • hit_rate
  • correctness
  • evidence_recall

Average Scores:
  • hit_rate: 0.569
  • correctness: 0.706
  • evidence_recall: 0.484

Run Evaluations:
  • avg_hit_rate: 0.569
  • avg_evidence_recall: 0.484
  • avg_correctness: 0.706

🔗 Experiment:
   https://cloud.langfuse.com/project/cmw4rolup00rlad0jzak94tgv/experiments/results?baseline=41f0ffd9585d5c6c


In [2]:
from langfuse import get_client
from finrag.eval_langhfuse import task, hit_rate, evidence_recall, correctness, average_scores

langfuse = get_client()
dataset = langfuse.get_dataset("financebench-dev")

result = dataset.run_experiment(
    name="filters-company-year",
    task=task,
    evaluators=[hit_rate, evidence_recall, correctness],
    run_evaluators=[average_scores],
    max_concurrency=1,
)

print(result.format())
langfuse.flush()

Individual Results: Hidden (51 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: filters-company-year
📋 Run name: filters-company-year - 2026-11-11T11:58:20.512207Z
51 items
Evaluations:
  • correctness
  • evidence_recall
  • hit_rate

Average Scores:
  • correctness: 0.745
  • evidence_recall: 0.533
  • hit_rate: 0.588

Run Evaluations:
  • avg_hit_rate: 0.588
  • avg_evidence_recall: 0.533
  • avg_correctness: 0.745

🔗 Experiment:
   https://cloud.langfuse.com/project/cmw4rolup00rlad0jzak94tgv/experiments/results?baseline=69849d998153844c


In [ ]:
#con rerank
from langfuse import get_client
from finrag.eval_langhfuse import task, hit_rate, evidence_recall, correctness, average_scores

langfuse = get_client()
dataset = langfuse.get_dataset("financebench-dev")

result = dataset.run_experiment(
    name="filters-company-year",
    task=task,
    evaluators=[hit_rate, evidence_recall, correctness],
    run_evaluators=[average_scores],
    max_concurrency=1,
)

print(result.format())
langfuse.flush()

In [1]:
import pandas as pd
from tqdm.auto import tqdm

from finrag.tools import extract_filter_from_query, search_pages
from finrag.ingestion import parse_doc_name

KS = [5, 10, 20]

df = pd.read_json("../data/processed/financebench_p.json", lines=True)
df_dev = df[df["split"] == "dev"].reset_index(drop=True)


def evidence_pages(evidence):
    return [[ev.get("doc_name") or ev.get("evidence_doc_name"), int(ev["evidence_page_num"])]
            for ev in evidence]

def hit_and_recall(retrieved, evidence):
    found = sum(1 for page in evidence if page in retrieved)
    return float(found > 0), found / len(evidence)

def classify(extracted, gold):
    if extracted is None:
        return "vuoto"
    return "corretto" if extracted == gold else "SBAGLIATO"


extraction_rows, results, wrong = [], [], []

for _, row in tqdm(df_dev.iterrows(), total=len(df_dev)):
    evidence = evidence_pages(row["evidence"])
    gold = parse_doc_name(row["doc_name"])

    # Filtri estratti dall'LLM
    extracted = extract_filter_from_query(row["question"]).model_dump()
    extraction_rows.append({
        "company": classify(extracted["company"], gold["company"]),
        "fiscal_year": classify(extracted["fiscal_year"], gold["fiscal_year"]),
    })

    # Salva i casi non corretti (sbagliati + vuoti) con il dettaglio
    for field in ["company", "fiscal_year"]:
        esito = classify(extracted[field], gold[field])
        if esito != "corretto":
            wrong.append({
                "esito": esito,
                "campo": field,
                "estratto": extracted[field],
                "giusto": gold[field],
                "doc_name": row["doc_name"],
                "domanda": row["question"],
            })

    setups = {
        "nessun filtro": {},
        "filtri estratti": extracted,
        "filtri perfetti": {"company": gold["company"], "fiscal_year": gold["fiscal_year"]},
    }
    for setup, filters in setups.items():
        pages = search_pages(row["question"], k=max(KS), **filters)
        retrieved = [[p["doc_name"], p["page_num"]] for p in pages]
        for k in KS:
            hit, recall = hit_and_recall(retrieved[:k], evidence)
            results.append({"setup": setup, "k": k, "hit_rate": hit, "evidence_recall": recall})


print("Accuratezza dell'estrazione (51 domande):")
print(pd.DataFrame(extraction_rows).apply(pd.Series.value_counts).fillna(0).astype(int), "\n")

print(
    pd.DataFrame(results)
    .groupby(["setup", "k"])[["hit_rate", "evidence_recall"]]
    .mean()
    .round(3)
)

print("\nCasi di estrazione non corretti:")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(wrong))


c:\Users\domen\Desktop\Agenti\Progetto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/51 [00:00<?, ?it/s]

Loading embeddinggemma2 on cuda 


100%|██████████| 51/51 [06:34<00:00,  7.73s/it]

Accuratezza dell'estrazione (51 domande):
           company  fiscal_year
SBAGLIATO        0            3
corretto        50           43
vuoto            1            5 

                    hit_rate  evidence_recall
setup           k                            
filtri estratti 5      0.569            0.513
                10     0.725            0.696
                20     0.784            0.765
filtri perfetti 5      0.627            0.572
                10     0.784            0.755
                20     0.824            0.804
nessun filtro   5      0.569            0.484
                10     0.725            0.660
                20     0.804            0.775

Casi di estrazione non corretti:


,esito,campo,estratto,giusto,doc_name,domanda
0,vuoto,fiscal_year,NaN,2023,3M_2023Q2_10Q,Does 3M maintain a stable trend of dividend distribution?
1,vuoto,fiscal_year,NaN,2022,AMCOR_2022_8K_dated-2022-07-01,What was the key agenda of the AMCOR's 8k filing dated 1st July 2022?
2,SBAGLIATO,fiscal_year,2023.0,2022,BOEING_2022_10K,What production rate changes is Boeing forecasting for FY2023?
3,vuoto,fiscal_year,NaN,2022,FOOTLOCKER_2022_8K_dated_2022-08-19,Does Foot Locker's new CEO have previous CEO experience in a similar company to Footlocker?
4,vuoto,company,NaN,FOOTLOCKER,FOOTLOCKER_2022_8K_dated-2022-05-20,Were there any board member nominees who had substantially more votes against joining than the other nominees?
5,vuoto,fiscal_year,NaN,2022,FOOTLOCKER_2022_8K_dated-2022-05-20,Were there any board member nominees who had substantially more votes against joining than the other nominees?
6,SBAGLIATO,fiscal_year,2023.0,2022,JOHNSON_JOHNSON_2022Q4_EARNINGS,Is growth in JnJ's adjusted EPS expected to accelerate in FY2023?
7,vuoto,fiscal_year,NaN,2022,JPMORGAN_2022_10K,"Are JPM's gross margins historically consistent (not fluctuating more than roughly 2% each year)? If gross margins are not a relevant metric for a company like this, then please state that and explain why."
8,SBAGLIATO,fiscal_year,2019.0,2021,PFIZER_2021_10K,Were there any potential events that are not in Pfizer's standard business operations that substantially increased net income in 2019?


In [1]:
import pandas as pd
from tqdm.auto import tqdm

from finrag.tools import extract_filter_from_query, search_pages
from finrag.ingestion import parse_doc_name

KS = [5, 10, 20]

df = pd.read_json("../data/processed/financebench_p.json", lines=True)
df_dev = df[df["split"] == "dev"].reset_index(drop=True)


def evidence_pages(evidence):
    return [[ev.get("doc_name") or ev.get("evidence_doc_name"), int(ev["evidence_page_num"])]
            for ev in evidence]

def hit_and_recall(retrieved, evidence):
    found = sum(1 for page in evidence if page in retrieved)
    return float(found > 0), found / len(evidence)

def classify(extracted, gold):
    if extracted is None:
        return "vuoto"
    return "corretto" if extracted == gold else "SBAGLIATO"


extraction_rows, results, wrong = [], [], []

for _, row in tqdm(df_dev.iterrows(), total=len(df_dev)):
    evidence = evidence_pages(row["evidence"])
    gold = parse_doc_name(row["doc_name"])

    # Filtri estratti dall'LLM
    extracted = extract_filter_from_query(row["question"]).model_dump()
    extraction_rows.append({
        "company": classify(extracted["company"], gold["company"]),
        "fiscal_year": classify(extracted["fiscal_year"], gold["fiscal_year"]),
    })

    # Salva i casi non corretti (sbagliati + vuoti) con il dettaglio
    for field in ["company", "fiscal_year"]:
        esito = classify(extracted[field], gold[field])
        if esito != "corretto":
            wrong.append({
                "esito": esito,
                "campo": field,
                "estratto": extracted[field],
                "giusto": gold[field],
                "doc_name": row["doc_name"],
                "domanda": row["question"],
            })

    setups = {
        "nessun filtro": {},
        "filtri estratti": extracted,
        "filtri perfetti": {"company": gold["company"], "fiscal_year": gold["fiscal_year"]},
    }
    for setup, filters in setups.items():
        pages = search_pages(row["question"], k=max(KS), **filters)
        retrieved = [[p["doc_name"], p["page_num"]] for p in pages]
        for k in KS:
            hit, recall = hit_and_recall(retrieved[:k], evidence)
            results.append({"setup": setup, "k": k, "hit_rate": hit, "evidence_recall": recall})


print("Accuratezza dell'estrazione (51 domande):")
print(pd.DataFrame(extraction_rows).apply(pd.Series.value_counts).fillna(0).astype(int), "\n")

print(
    pd.DataFrame(results)
    .groupby(["setup", "k"])[["hit_rate", "evidence_recall"]]
    .mean()
    .round(3)
)

print("\nCasi di estrazione non corretti:")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(wrong))


c:\Users\domen\Desktop\Agenti\Progetto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/51 [00:00<?, ?it/s]

Loading embeddinggemma2 on cuda 


100%|██████████| 51/51 [07:51<00:00,  9.25s/it]

Accuratezza dell'estrazione (51 domande):
           company  fiscal_year
SBAGLIATO        0            1
corretto        50           46
vuoto            1            4 

                    hit_rate  evidence_recall
setup           k                            
filtri estratti 5      0.608            0.552
                10     0.765            0.735
                20     0.824            0.804
filtri perfetti 5      0.627            0.572
                10     0.784            0.755
                20     0.824            0.804
nessun filtro   5      0.569            0.484
                10     0.725            0.660
                20     0.804            0.775

Casi di estrazione non corretti:


,esito,campo,estratto,giusto,doc_name,domanda
0,vuoto,fiscal_year,NaN,2023,3M_2023Q2_10Q,Does 3M maintain a stable trend of dividend distribution?
1,vuoto,fiscal_year,NaN,2022,FOOTLOCKER_2022_8K_dated_2022-08-19,Does Foot Locker's new CEO have previous CEO experience in a similar company to Footlocker?
2,vuoto,company,NaN,FOOTLOCKER,FOOTLOCKER_2022_8K_dated-2022-05-20,Were there any board member nominees who had substantially more votes against joining than the other nominees?
3,vuoto,fiscal_year,NaN,2022,FOOTLOCKER_2022_8K_dated-2022-05-20,Were there any board member nominees who had substantially more votes against joining than the other nominees?
4,vuoto,fiscal_year,NaN,2022,JPMORGAN_2022_10K,"Are JPM's gross margins historically consistent (not fluctuating more than roughly 2% each year)? If gross margins are not a relevant metric for a company like this, then please state that and explain why."
5,SBAGLIATO,fiscal_year,2019.0,2021,PFIZER_2021_10K,Were there any potential events that are not in Pfizer's standard business operations that substantially increased net income in 2019?


In [5]:
from finrag.config import get_qdrant, COLLECTION
from qdrant_client import models

client = get_qdrant()


In [6]:
client.count(
    collection_name=COLLECTION,
    count_filter=models.Filter(must=[
        models.FieldCondition(key="company", match=models.MatchValue(value="PFIZER")),
        models.FieldCondition(key="fiscal_year", match=models.MatchValue(value=2019)),
    ]),
    exact=True,
)

CountResult(count=233)